In [ ]:
import sys
from pathlib import Path

print("Python 버전:", sys.version)
print("현재 작업 폴더:", Path.cwd())

In [ ]:
from pathlib import Path

BASE_DIR = Path.cwd()

AIHUB_DIR = BASE_DIR / "118.안면 인식 에이징(aging) 이미지 데이터"

TRAIN_SRC_ZIP_DIR = AIHUB_DIR / "Training" / "01.원천데이터"
TRAIN_LAB_ZIP_DIR = AIHUB_DIR / "Training" / "02.라벨링데이터"

VAL_SRC_ZIP_DIR = AIHUB_DIR / "Validation" / "01.원천데이터"
VAL_LAB_ZIP_DIR = AIHUB_DIR / "Validation" / "02.라벨링데이터"

paths = {
    "프로젝트 폴더": BASE_DIR,
    "AI Hub 데이터 폴더": AIHUB_DIR,
    "Training 원천데이터": TRAIN_SRC_ZIP_DIR,
    "Training 라벨링데이터": TRAIN_LAB_ZIP_DIR,
    "Validation 원천데이터": VAL_SRC_ZIP_DIR,
    "Validation 라벨링데이터": VAL_LAB_ZIP_DIR,
}

for name, path in paths.items():
    print(f"{name}:")
    print(path)
    print("존재 여부:", path.exists())
    print("-" * 80)

In [ ]:
from pathlib import Path

BASE_DIR = Path.cwd()
AIHUB_DIR = BASE_DIR / "118.안면 인식 에이징(aging) 이미지 데이터"

print("AIHUB_DIR:", AIHUB_DIR)
print("존재:", AIHUB_DIR.exists())

print("\n[AIHUB_DIR 바로 아래 실제 이름]")
for p in AIHUB_DIR.iterdir():
    print(repr(p.name), "폴더:", p.is_dir())

In [ ]:
from pathlib import Path

BASE_DIR = Path.cwd()

AIHUB_DIR = BASE_DIR / "118.안면 인식 에이징(aging) 이미지 데이터"
OPEN_DIR = AIHUB_DIR / "01-1.정식개방데이터"

TRAIN_SRC_ZIP_DIR = OPEN_DIR / "Training" / "01.원천데이터"
TRAIN_LAB_ZIP_DIR = OPEN_DIR / "Training" / "02.라벨링데이터"

VAL_SRC_ZIP_DIR = OPEN_DIR / "Validation" / "01.원천데이터"
VAL_LAB_ZIP_DIR = OPEN_DIR / "Validation" / "02.라벨링데이터"

paths = {
    "OPEN_DIR": OPEN_DIR,
    "Training 원천데이터": TRAIN_SRC_ZIP_DIR,
    "Training 라벨링데이터": TRAIN_LAB_ZIP_DIR,
    "Validation 원천데이터": VAL_SRC_ZIP_DIR,
    "Validation 라벨링데이터": VAL_LAB_ZIP_DIR,
}

for name, path in paths.items():
    print(name)
    print(path)
    print("존재:", path.exists())
    print("-" * 80)

In [ ]:
archive_exts = [".zip", ".7z", ".alz", ".egg"]

def list_archives(folder: Path):
    return [
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in archive_exts
    ]

train_src_archives = list_archives(TRAIN_SRC_ZIP_DIR)
train_lab_archives = list_archives(TRAIN_LAB_ZIP_DIR)
val_src_archives = list_archives(VAL_SRC_ZIP_DIR)
val_lab_archives = list_archives(VAL_LAB_ZIP_DIR)

print("Training 원천 압축파일:", len(train_src_archives))
print("Training 라벨 압축파일:", len(train_lab_archives))
print("Validation 원천 압축파일:", len(val_src_archives))
print("Validation 라벨 압축파일:", len(val_lab_archives))

print("\nTraining 원천 예시:")
for p in train_src_archives[:10]:
    print(p.name)

print("\nTraining 라벨 예시:")
for p in train_lab_archives[:10]:
    print(p.name)

In [ ]:
import zipfile
import subprocess
import shutil
from pathlib import Path

# 압축 해제 결과 저장 위치
DATA_DIR = BASE_DIR / "data" / "aging_face"

TRAIN_IMAGES_DIR = DATA_DIR / "train" / "images"
TRAIN_LABELS_DIR = DATA_DIR / "train" / "labels"
VAL_IMAGES_DIR = DATA_DIR / "val" / "images"
VAL_LABELS_DIR = DATA_DIR / "val" / "labels"

for d in [TRAIN_IMAGES_DIR, TRAIN_LABELS_DIR, VAL_IMAGES_DIR, VAL_LABELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("압축 해제 결과 저장 폴더 생성 완료")
print(TRAIN_IMAGES_DIR)
print(TRAIN_LABELS_DIR)
print(VAL_IMAGES_DIR)
print(VAL_LABELS_DIR)

In [ ]:
import zipfile
import subprocess
import shutil
from pathlib import Path

def find_7z_exe():
    """
    7-Zip 실행 파일 위치 찾기
    """
    candidates = [
        r"C:\Program Files\7-Zip\7z.exe",
        r"C:\Program Files (x86)\7-Zip\7z.exe",
    ]

    for c in candidates:
        p = Path(c)
        if p.exists():
            return str(p)

    found = shutil.which("7z")
    if found:
        return found

    return None


SEVEN_ZIP = find_7z_exe()
print("7-Zip 경로:", SEVEN_ZIP)


def safe_folder_name(path: Path) -> str:
    """
    압축파일 이름을 폴더명으로 안전하게 변환
    예: TS_0001.zip -> TS_0001
    """
    return path.stem.replace(" ", "_").replace(".", "_")


def extract_zip_with_python(archive_path: Path, dest_dir: Path):
    """
    zip 파일은 파이썬 기본 zipfile로 먼저 압축 해제
    """
    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall(dest_dir)


def extract_with_7z(archive_path: Path, dest_dir: Path):
    """
    zipfile로 실패하거나 7z/alz/egg 같은 파일은 7-Zip으로 압축 해제
    """
    if SEVEN_ZIP is None:
        raise RuntimeError(
            "7-Zip을 찾을 수 없습니다. zipfile로 실패하면 7-Zip 설치가 필요합니다."
        )

    cmd = [
        SEVEN_ZIP,
        "x",
        str(archive_path),
        f"-o{dest_dir}",
        "-y"
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="ignore"
    )

    if result.returncode != 0:
        print("STDOUT:", result.stdout[-1000:])
        print("STDERR:", result.stderr[-1000:])
        raise RuntimeError(f"압축 해제 실패: {archive_path.name}")


def extract_archive(archive_path: Path, dest_base_dir: Path):
    """
    압축파일 1개를 압축파일 이름별 폴더에 해제한다.
    이미 해제한 파일은 _EXTRACT_DONE.txt를 보고 건너뛴다.
    """
    out_dir = dest_base_dir / safe_folder_name(archive_path)
    done_marker = out_dir / "_EXTRACT_DONE.txt"

    if done_marker.exists():
        print("이미 해제됨:", archive_path.name)
        return "skipped"

    out_dir.mkdir(parents=True, exist_ok=True)

    try:
        if archive_path.suffix.lower() == ".zip":
            try:
                extract_zip_with_python(archive_path, out_dir)
            except Exception:
                print("zipfile 실패 → 7-Zip 재시도:", archive_path.name)
                extract_with_7z(archive_path, out_dir)
        else:
            extract_with_7z(archive_path, out_dir)

        done_marker.write_text(
            f"Extracted from: {archive_path}\n",
            encoding="utf-8"
        )

        print("완료:", archive_path.name)
        return "success"

    except Exception as e:
        print("실패:", archive_path.name)
        print("오류:", e)
        return f"failed: {e}"


def extract_many(archives, dest_base_dir: Path, title: str):
    """
    압축파일 여러 개를 순서대로 해제
    """
    print("\n" + "=" * 100)
    print(title)
    print("압축파일 수:", len(archives))
    print("해제 위치:", dest_base_dir)

    log = []

    for idx, archive_path in enumerate(archives, start=1):
        print("\n" + "-" * 80)
        print(f"[{idx}/{len(archives)}] {archive_path.name}")

        status = extract_archive(archive_path, dest_base_dir)
        log.append((archive_path.name, status))

    return log

In [ ]:
test_log_train_src = extract_many(
    train_src_archives[:1],
    TRAIN_IMAGES_DIR,
    "테스트: Training 원천데이터 1개"
)

test_log_train_lab = extract_many(
    train_lab_archives[:1],
    TRAIN_LABELS_DIR,
    "테스트: Training 라벨링데이터 1개"
)

In [ ]:
from collections import Counter

def count_extensions(root: Path):
    counter = Counter()
    total = 0

    for p in root.rglob("*"):
        if p.is_file() and p.name != "_EXTRACT_DONE.txt":
            counter[p.suffix.lower()] += 1
            total += 1

    return total, counter


for name, root in [
    ("train images", TRAIN_IMAGES_DIR),
    ("train labels", TRAIN_LABELS_DIR),
]:
    total, counter = count_extensions(root)

    print("\n" + "=" * 80)
    print(name)
    print("전체 파일 수:", total)
    print("확장자별 개수:")
    for ext, cnt in counter.most_common(20):
        print(ext, cnt)

In [ ]:
log_train_src = extract_many(
    train_src_archives,
    TRAIN_IMAGES_DIR,
    "전체: Training 원천데이터 → train/images"
)

log_train_lab = extract_many(
    train_lab_archives,
    TRAIN_LABELS_DIR,
    "전체: Training 라벨링데이터 → train/labels"
)

log_val_src = extract_many(
    val_src_archives,
    VAL_IMAGES_DIR,
    "전체: Validation 원천데이터 → val/images"
)

log_val_lab = extract_many(
    val_lab_archives,
    VAL_LABELS_DIR,
    "전체: Validation 라벨링데이터 → val/labels"
)

print("전체 압축 해제 완료")

In [ ]:
for name, root in [
    ("train images", TRAIN_IMAGES_DIR),
    ("train labels", TRAIN_LABELS_DIR),
    ("val images", VAL_IMAGES_DIR),
    ("val labels", VAL_LABELS_DIR),
]:
    total, counter = count_extensions(root)

    print("\n" + "=" * 80)
    print(name)
    print("전체 파일 수:", total)
    print("확장자별 개수:")
    for ext, cnt in counter.most_common(20):
        print(ext, cnt)

In [ ]:
import json
from pathlib import Path
from pprint import pprint

# train labels 폴더에서 JSON 파일 5개 가져오기
sample_json_files = list(TRAIN_LABELS_DIR.rglob("*.json"))[:5]

print("샘플 JSON 개수:", len(sample_json_files))

for json_path in sample_json_files:
    print("\n" + "=" * 100)
    print("JSON 파일:", json_path)
    
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    pprint(data)

In [ ]:
sample_image_files = list(TRAIN_IMAGES_DIR.rglob("*.png"))[:10]

print("샘플 이미지 개수:", len(sample_image_files))

for p in sample_image_files:
    print(p)

In [ ]:
from pathlib import Path
import json

# 샘플 JSON 하나 선택
sample_json_path = list(TRAIN_LABELS_DIR.rglob("*.json"))[0]

with open(sample_json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

filename = data["filename"]
fmt = data["format"]
image_filename = f"{filename}.{fmt}"

print("JSON 파일:", sample_json_path)
print("JSON 안 filename:", filename)
print("예상 이미지 파일명:", image_filename)

# train images 전체에서 같은 이름의 이미지 찾기
matched_images = list(TRAIN_IMAGES_DIR.rglob(image_filename))

print("매칭된 이미지 개수:", len(matched_images))

for p in matched_images[:5]:
    print(p)

In [ ]:
import json
import pandas as pd
from pathlib import Path

def make_age_group(age):
    """
    나이를 나이대로 변환
    """
    if age < 10:
        return "0대"
    elif age < 20:
        return "10대"
    elif age < 30:
        return "20대"
    elif age < 40:
        return "30대"
    elif age < 50:
        return "40대"
    elif age < 60:
        return "50대"
    else:
        return "60대이상"


def build_dataset_csv(images_dir: Path, labels_dir: Path, split: str, output_csv: Path):
    """
    labels_dir의 JSON을 읽고,
    images_dir에서 해당 PNG 이미지를 찾아 dataset CSV 생성
    """
    json_files = list(labels_dir.rglob("*.json"))
    print(f"{split} JSON 파일 수:", len(json_files))

    rows = []
    missing_images = []

    for idx, json_path in enumerate(json_files, start=1):
        if idx % 5000 == 0:
            print(f"{idx}/{len(json_files)} 처리 중...")

        try:
            with open(json_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            filename = data.get("filename")
            fmt = data.get("format", "png")
            image_filename = f"{filename}.{fmt}"

            # 이미지 찾기
            matched = list(images_dir.rglob(image_filename))

            if len(matched) == 0:
                missing_images.append(str(json_path))
                continue

            image_path = matched[0]

            age_past = data.get("age_past")
            age_now = data.get("age_now")
            gender = data.get("gender")
            person_id = data.get("id")
            birth = data.get("birth")
            device = data.get("device")

            rows.append({
                "image_path": str(image_path),
                "json_path": str(json_path),
                "filename": filename,
                "format": fmt,
                "age_past": age_past,
                "age_now": age_now,
                "age_group": make_age_group(age_past),
                "gender": gender,
                "person_id": person_id,
                "birth": birth,
                "device": device,
                "split": split
            })

        except Exception as e:
            print("오류 발생:", json_path, e)

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False, encoding="utf-8-sig")

    print("\n완료")
    print("저장 위치:", output_csv)
    print("생성된 행 수:", len(df))
    print("매칭 실패 JSON 수:", len(missing_images))

    return df, missing_images

In [ ]:
import sys

print(sys.executable)

In [ ]:
import sys
!{sys.executable} -m pip install pandas

In [ ]:
import pandas as pd

print(pd.__version__)

In [ ]:
import sys
!{sys.executable} -m pip install pandas pillow matplotlib scikit-learn

In [ ]:
import pandas as pd
import PIL
import matplotlib
import sklearn

print("pandas:", pd.__version__)
print("Pillow:", PIL.__version__)
print("matplotlib:", matplotlib.__version__)
print("sklearn:", sklearn.__version__)

In [ ]:
from pathlib import Path

def build_image_index(images_dir: Path):
    image_exts = [".png", ".jpg", ".jpeg"]
    image_files = [
        p for p in images_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in image_exts
    ]

    image_index = {}
    duplicate_stems = []

    for p in image_files:
        stem = p.stem

        if stem in image_index:
            duplicate_stems.append(stem)
        else:
            image_index[stem] = p

    print("이미지 파일 수:", len(image_files))
    print("인덱스 개수:", len(image_index))
    print("중복 파일명 수:", len(duplicate_stems))

    return image_index, duplicate_stems


train_image_index, train_duplicate_stems = build_image_index(TRAIN_IMAGES_DIR)
val_image_index, val_duplicate_stems = build_image_index(VAL_IMAGES_DIR)

In [ ]:
import json
import pandas as pd
from pathlib import Path

def make_age_group(age):
    if age is None:
        return None

    age = int(age)

    if age < 10:
        return "0대"
    elif age < 20:
        return "10대"
    elif age < 30:
        return "20대"
    elif age < 40:
        return "30대"
    elif age < 50:
        return "40대"
    elif age < 60:
        return "50대"
    else:
        return "60대이상"


def build_dataset_csv_fast(labels_dir: Path, image_index: dict, split: str, output_csv: Path):
    json_files = list(labels_dir.rglob("*.json"))
    print(f"{split} JSON 파일 수:", len(json_files))

    rows = []
    missing_images = []
    error_files = []

    for idx, json_path in enumerate(json_files, start=1):
        if idx % 5000 == 0:
            print(f"{idx}/{len(json_files)} 처리 중...")

        try:
            with open(json_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            filename = data.get("filename")
            image_path = image_index.get(filename)

            if image_path is None:
                missing_images.append(str(json_path))
                continue

            age_past = data.get("age_past")
            age_now = data.get("age_now")

            rows.append({
                "image_path": str(image_path),
                "json_path": str(json_path),
                "filename": filename,
                "format": data.get("format"),
                "age_past": age_past,
                "age_now": age_now,
                "age_group": make_age_group(age_past),
                "gender": data.get("gender"),
                "person_id": data.get("id"),
                "birth": data.get("birth"),
                "device": data.get("device"),
                "width": data.get("width"),
                "height": data.get("height"),
                "imgsize": data.get("imgsize"),
                "split": split
            })

        except Exception as e:
            error_files.append((str(json_path), str(e)))

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False, encoding="utf-8-sig")

    print("\n완료:", split)
    print("CSV 저장 위치:", output_csv)
    print("생성된 행 수:", len(df))
    print("이미지 매칭 실패 수:", len(missing_images))
    print("JSON 처리 오류 수:", len(error_files))

    return df, missing_images, error_files

In [ ]:
TRAIN_CSV = DATA_DIR / "train_dataset.csv"
VAL_CSV = DATA_DIR / "val_dataset.csv"

train_df, train_missing, train_errors = build_dataset_csv_fast(
    labels_dir=TRAIN_LABELS_DIR,
    image_index=train_image_index,
    split="train",
    output_csv=TRAIN_CSV
)

val_df, val_missing, val_errors = build_dataset_csv_fast(
    labels_dir=VAL_LABELS_DIR,
    image_index=val_image_index,
    split="val",
    output_csv=VAL_CSV
)

In [ ]:
print("train 매칭 실패 수:", len(train_missing))
print("val 매칭 실패 수:", len(val_missing))

print("\ntrain 매칭 실패 예시:")
for p in train_missing[:20]:
    print(p)

print("\nval 매칭 실패 예시:")
for p in val_missing[:20]:
    print(p)

In [ ]:
def show_files_in_folder(folder: Path, n=30):
    print("폴더:", folder)
    print("존재:", folder.exists())
    
    files = [p for p in folder.rglob("*") if p.is_file()]
    print("파일 수:", len(files))
    
    for p in files[:n]:
        print(p.name)

show_files_in_folder(TRAIN_IMAGES_DIR / "TS_0039", n=60)
show_files_in_folder(TRAIN_IMAGES_DIR / "TS_0047", n=60)
show_files_in_folder(VAL_IMAGES_DIR / "VS_0929", n=60)

In [ ]:
import json
from pathlib import Path

def show_missing_expected_files(missing_list, n=20):
    for json_path_str in missing_list[:n]:
        json_path = Path(json_path_str)
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        expected = data["filename"] + "." + data.get("format", "png")
        print("JSON:", json_path.name)
        print("예상 이미지:", expected)
        print("-" * 80)

print("Train missing expected")
show_missing_expected_files(train_missing, n=20)

print("\nVal missing expected")
show_missing_expected_files(val_missing, n=20)

In [ ]:
def count_suffix_type_from_json(labels_dir: Path):
    counts = {}
    
    for json_path in labels_dir.rglob("*.json"):
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        filename = data.get("filename", "")
        suffix_type = filename.split("_")[-1]  # F 또는 D 예상
        counts[suffix_type] = counts.get(suffix_type, 0) + 1
    
    return counts


def count_suffix_type_from_images(images_dir: Path):
    counts = {}
    
    for img_path in images_dir.rglob("*.png"):
        filename = img_path.stem
        suffix_type = filename.split("_")[-1]
        counts[suffix_type] = counts.get(suffix_type, 0) + 1
    
    return counts


print("Train JSON F/D 분포:", count_suffix_type_from_json(TRAIN_LABELS_DIR))
print("Train Image F/D 분포:", count_suffix_type_from_images(TRAIN_IMAGES_DIR))

print("\nVal JSON F/D 분포:", count_suffix_type_from_json(VAL_LABELS_DIR))
print("Val Image F/D 분포:", count_suffix_type_from_images(VAL_IMAGES_DIR))

In [ ]:
print("Train 나이대 분포")
print(train_df["age_group"].value_counts().sort_index())

print("\nValidation 나이대 분포")
print(val_df["age_group"].value_counts().sort_index())

print("\nTrain 성별 분포")
print(train_df["gender"].value_counts())

print("\nValidation 성별 분포")
print(val_df["gender"].value_counts())

In [ ]:
# filename 정규화 함수
from pathlib import Path
import json
import pandas as pd

def normalize_filename(name):
    """
    JSON의 filename 값을 이미지 stem과 매칭하기 좋게 정리한다.
    예:
    xxx_F.png -> xxx_F
    xxx_f     -> xxx_F
    xxx_d     -> xxx_D
    """
    if name is None:
        return None

    name = str(name).strip()

    # 확장자가 들어간 경우 제거
    name = Path(name).stem

    parts = name.split("_")

    # 마지막 토큰이 f/d면 대문자로 통일
    if len(parts) > 0 and parts[-1].lower() in ["f", "d"]:
        parts[-1] = parts[-1].upper()

    return "_".join(parts)

In [ ]:
# 이미지 인덱스도 정규화해서 다시 만들기
def build_image_index_normalized(images_dir: Path):
    image_exts = [".png", ".jpg", ".jpeg"]
    image_files = [
        p for p in images_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in image_exts
    ]

    image_index = {}
    duplicate_keys = []

    for p in image_files:
        key = normalize_filename(p.stem)

        if key in image_index:
            duplicate_keys.append(key)
        else:
            image_index[key] = p

    print("이미지 파일 수:", len(image_files))
    print("인덱스 개수:", len(image_index))
    print("중복 key 수:", len(duplicate_keys))

    return image_index, duplicate_keys


train_image_index_norm, train_dup_norm = build_image_index_normalized(TRAIN_IMAGES_DIR)
val_image_index_norm, val_dup_norm = build_image_index_normalized(VAL_IMAGES_DIR)

In [ ]:
# 3. dataset CSV 다시 만들기
def make_age_group(age):
    if age is None:
        return None

    age = int(age)

    if age < 10:
        return "0대"
    elif age < 20:
        return "10대"
    elif age < 30:
        return "20대"
    elif age < 40:
        return "30대"
    elif age < 50:
        return "40대"
    elif age < 60:
        return "50대"
    else:
        return "60대이상"


def build_dataset_csv_normalized(labels_dir: Path, image_index: dict, split: str, output_csv: Path):
    json_files = list(labels_dir.rglob("*.json"))
    print(f"{split} JSON 파일 수:", len(json_files))

    rows = []
    missing_images = []
    error_files = []

    for idx, json_path in enumerate(json_files, start=1):
        if idx % 5000 == 0:
            print(f"{idx}/{len(json_files)} 처리 중...")

        try:
            with open(json_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            raw_filename = data.get("filename")
            filename_key = normalize_filename(raw_filename)

            image_path = image_index.get(filename_key)

            if image_path is None:
                missing_images.append(str(json_path))
                continue

            age_past = data.get("age_past")
            age_now = data.get("age_now")

            rows.append({
                "image_path": str(image_path),
                "json_path": str(json_path),
                "filename_raw": raw_filename,
                "filename_key": filename_key,
                "format": data.get("format"),
                "age_past": age_past,
                "age_now": age_now,
                "age_group": make_age_group(age_past),
                "gender": data.get("gender"),
                "person_id": data.get("id"),
                "birth": data.get("birth"),
                "device": data.get("device"),
                "width": data.get("width"),
                "height": data.get("height"),
                "imgsize": data.get("imgsize"),
                "split": split
            })

        except Exception as e:
            error_files.append((str(json_path), str(e)))

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False, encoding="utf-8-sig")

    print("\n완료:", split)
    print("CSV 저장 위치:", output_csv)
    print("생성된 행 수:", len(df))
    print("이미지 매칭 실패 수:", len(missing_images))
    print("JSON 처리 오류 수:", len(error_files))

    return df, missing_images, error_files

In [ ]:
TRAIN_CSV = DATA_DIR / "train_dataset.csv"
VAL_CSV = DATA_DIR / "val_dataset.csv"

train_df, train_missing, train_errors = build_dataset_csv_normalized(
    labels_dir=TRAIN_LABELS_DIR,
    image_index=train_image_index_norm,
    split="train",
    output_csv=TRAIN_CSV
)

val_df, val_missing, val_errors = build_dataset_csv_normalized(
    labels_dir=VAL_LABELS_DIR,
    image_index=val_image_index_norm,
    split="val",
    output_csv=VAL_CSV
)

In [ ]:
print("train_df 크기:", train_df.shape)
print("val_df 크기:", val_df.shape)

print("\nTrain 나이대 분포")
print(train_df["age_group"].value_counts().sort_index())

print("\nValidation 나이대 분포")
print(val_df["age_group"].value_counts().sort_index())

print("\nTrain 성별 분포")
print(train_df["gender"].value_counts())

print("\nValidation 성별 분포")
print(val_df["gender"].value_counts())

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

sample = train_df.sample(9, random_state=42)

plt.figure(figsize=(10, 10))

for i, (_, row) in enumerate(sample.iterrows(), start=1):
    img = Image.open(row["image_path"]).convert("RGB")
    
    plt.subplot(3, 3, i)
    plt.imshow(img)
    plt.title(f'age:{row["age_past"]}, group:{row["age_group"]}, {row["gender"]}')
    plt.axis("off")

plt.tight_layout()
plt.show()

In [1]:
import torch
import torchvision

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())

torch: 2.12.0+cpu
torchvision: 0.27.0+cpu
CUDA 사용 가능: False
